# Case Sleeping Beauty — the B index and awakening time

Ke, Ferrara, Radicchi & Flammini's "beauty coefficient". A sleeping beauty is a case that is
ignored for years and then cited heavily — the shape of its citation curve, not its total.

## The metric

Let $c_t$ be citations received at age $t$ (years since decision), $t_m$ the age of the
citation peak, and $c_m = c_{t_m}$. Then

$$B = \sum_{t=0}^{t_m} \frac{\frac{c_m - c_0}{t_m}\,t + c_0 - c_t}{\max(c_t, 1)}$$

which measures how far the real curve sits below the straight line from year 0 to the peak. A
case that peaks immediately has $t_m = 0$ and is **not** a sleeping beauty — it gets `B = 0`,
`T = 0` by definition rather than by a missing value.

`T` is the awakening time: the age at maximum distance from that line — when the case woke up.

## Output
`Case law/output/case_sb.parquet` — `case_id, SB_B, SB_T, n_cite`

Only cases with at least one dated citation appear; `n_cite` is how many citations the curve was
built from. `B` is meaningless on two or three citations, so filter on `n_cite`.

## Implementation
`numba`, over a per-case CSR of (age, count) pairs — the same kernel as
`PatentView/notebook/patent_sb.ipynb`, so the two are the same estimator on different graphs.

In [1]:
%%time
import os, sys, gc
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Case law')
import cl_common as cl
OUT_FP = cl.out('case_sb.parquet')
cl.preflight('case_sb')

c_from, c_to, year, uni = cl.load_graph()
n = len(uni)
yF, yT = year[c_from], year[c_to]
ok = (yF > 0) & (yT > 0) & (yF >= yT)
cited = c_to[ok].astype(np.int64)
age = (yF[ok] - yT[ok]).astype(np.int32)
print(f'edges {len(c_from):,}  ->  usable {len(cited):,}   max age {age.max()}')

# (case, age) -> count, as a CSR keyed by case. One sort, no hash over 47M rows.
key = cited * 1024 + age                      # ages fit in 1024 (max observed printed above)
assert age.max() < 1024
uk, cnt = np.unique(key, return_counts=True)
del key; gc.collect()
w_of = (uk // 1024).astype(np.int64)
ages  = (uk % 1024).astype(np.int32)
cnts  = cnt.astype(np.int64)
ptr = np.zeros(n + 1, np.int64)
np.add.at(ptr, w_of + 1, 1)
np.cumsum(ptr, out=ptr)
print(f'(case, age) pairs: {len(uk):,}   cases with >=1 citation: {int((np.diff(ptr) > 0).sum()):,}')

case law : /project/jevans/Dawoon/Science of Science/Case law
output   : /project/jevans/Dawoon/Science of Science/Case law/output
cache    : /project/jevans/Dawoon/Science of Science/Case law/cache

  case_sb                     OK
graph cache present: /project/jevans/Dawoon/Science of Science/Case law/cache/case_graph.npz
edges 47,519,638  ->  usable 47,519,638   max age 320
(case, age) pairs: 25,816,136   cases with >=1 citation: 3,787,861


In [2]:
from numba import njit, prange

@njit
def _sb_one(C):
    """C[age] = citations received at that age. Returns (B, T). Mirrors patent_sb / MAG-SB."""
    m = len(C)
    t_m = 0; cmax = C[0]
    for t in range(m):
        if C[t] > cmax:
            cmax = C[t]; t_m = t
    if t_m == 0:
        return 0.0, 0            # peaks at decision -> not a sleeping beauty
    c_m = C[t_m]; c_0 = C[0]
    B = 0.0
    for t in range(t_m + 1):
        den = C[t] if C[t] != 0.0 else 1.0
        B += ((c_m - c_0) / t_m * t + c_0 - C[t]) / den
    norm = np.sqrt((c_m - c_0) ** 2 + t_m * t_m)
    T = 0; dmax = -1.0
    for t in range(t_m + 1):
        d = abs((c_m - c_0) * t + (c_0 - C[t]) * t_m) / norm
        if d > dmax:
            dmax = d; T = t
    return B, T


@njit(parallel=True)
def sb_from_age_csr(ptr, ages, cnts, SBB, SBT, NC):
    n = len(ptr) - 1
    for w in prange(n):
        a0 = ptr[w]; a1 = ptr[w + 1]
        if a1 == a0:
            continue
        mx = -1; tot = 0
        for j in range(a0, a1):
            ag = ages[j]
            if ag >= 0:
                tot += cnts[j]
                if ag > mx:
                    mx = ag
        if mx < 0:
            continue
        C = np.zeros(mx + 1, np.float64)
        for j in range(a0, a1):
            ag = ages[j]
            if ag >= 0:
                C[ag] += cnts[j]
        B, T = _sb_one(C)
        SBB[w] = B; SBT[w] = T; NC[w] = tot


print('numba kernel ready')

numba kernel ready


In [3]:
%%time
SBB = np.full(n, np.nan, np.float64); SBT = np.full(n, -1, np.int32); NC = np.zeros(n, np.int64)
sb_from_age_csr(ptr, ages, cnts, SBB, SBT, NC)
has = NC > 0
sb = pd.DataFrame({'case_id': uni[has], 'SB_B': SBB[has].astype(np.float32),
                   'SB_T': SBT[has].astype(np.int16), 'n_cite': NC[has].astype(np.int32)})
sb.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(sb):,} rows, {os.path.getsize(OUT_FP)/1e6:.0f} MB)')
print(f'  SB_B  mean {sb.SB_B.mean():.3f}  median {sb.SB_B.median():.3f}  '
      f'p99 {sb.SB_B.quantile(.99):.2f}  max {sb.SB_B.max():.1f}')
print(f'  SB_T  median {int(sb.SB_T.median())}  p99 {int(sb.SB_T.quantile(.99))}  max {sb.SB_T.max()}')
print(f'  B == 0 (peaks at decision): {int((sb.SB_B == 0).sum()):,}  '
      f'({(sb.SB_B == 0).mean()*100:.1f}%)')
print('\ntop sleeping beauties with a real citation record (n_cite >= 50):')
display(sb[sb.n_cite >= 50].nlargest(10, 'SB_B'))

WROTE /project/jevans/Dawoon/Science of Science/Case law/output/case_sb.parquet  (3,787,861 rows, 30 MB)
  SB_B  mean 6.007  median 1.500  p99 60.00  max 4682.8
  SB_T  median 2  p99 63  max 245
  B == 0 (peaks at decision): 1,119,581  (29.6%)

top sleeping beauties with a real citation record (n_cite >= 50):


,case_id,SB_B,SB_T,n_cite
3515868,10591698,4682.804688,23,1314
3723300,12121367,3454.337891,162,492
3723398,12121622,3300.233887,159,2878
2835541,8294520,3153.725098,106,335
2537742,6627259,2825.310059,177,223
3677458,11749315,2499.779297,184,321
872946,1421261,2233.148193,145,1219
2501506,6142415,2028.850952,70,2713
876736,1427693,1635.648682,158,1176
3675662,11720199,1613.138794,57,3000
